# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# **Read the secret**

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
from huggingface_hub import login

login(token=HF_TOKEN)

**inspect the dataset**

In [3]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

# Download the March 2026 Parquet file

In [4]:
from huggingface_hub import hf_hub_download

parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(parquet_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [5]:
import pandas as pd

df = pd.read_parquet(parquet_path)
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis + Time Window

One row: The daily performance of one content item (content_hash_id) for one client (client_hash_id) on one reporting date (report_date).

Table: fact_content_daily_performance

Time Window: March 2026 (month=2026-03), following the assignment recommendation to use a mid-panel month rather than the final month.

Purpose: This table provides page-level search visibility, traffic, and engagement signals that are appropriate for Ranking Signal Analysis.

In [6]:
df["report_date"] = pd.to_datetime(df["report_date"])

print("Rows, Columns:", df.shape)

print("Columns:")
print(df.columns.tolist())

duplicates = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("Duplicate rows:", duplicates)

print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

Rows, Columns: (9841378, 30)
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Duplicate rows: 0
Start date: 2026-03-01 00:00:00
End date: 2026-03-31 00:00:00


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature Fields

These fields describe search visibility, user engagement, and website traffic.

gsc_impressions
gsc_clicks
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
scroll_events
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
Label / Proxy

The dataset does not contain an explicit optimization-priority label. Therefore, a proxy relevance score will be engineered in a later notebook using observed search performance signals. This engineered score will serve as the target for training and evaluating the ranking model.

Context
report_date
client_hash_id
content_hash_id

These fields uniquely identify each observation but are not used as predictive features.

Excluded
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other

Reason

These variables measure AI referral traffic rather than traditional organic search performance, which is outside the scope of this project.

In [7]:
# Organize fields into the four buckets

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "scroll_events",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid"
]

label = [
    "Proxy Relevance Score (engineered later)"
]

context = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

excluded = [
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other"
]

In [8]:
import pandas as pd

field_groups = pd.DataFrame({
    "Feature": pd.Series(features),
    "Label/Proxy": pd.Series(label),
    "Context": pd.Series(context),
    "Excluded": pd.Series(excluded)
})

field_groups


,Feature,Label/Proxy,Context,Excluded
0,gsc_impressions,Proxy Relevance Score (engineered later),report_date,sessions_ai
1,gsc_clicks,NaN,client_hash_id,ai_chatgpt
2,gsc_avg_position,NaN,content_hash_id,ai_perplexity
3,ga4_pageviews,NaN,NaN,ai_gemini
4,ga4_sessions,NaN,NaN,ai_copilot
5,ga4_users,NaN,NaN,ai_claude
6,ga4_engaged_sessions,NaN,NaN,ai_meta
7,ga4_total_engagement_sec,NaN,NaN,ai_other
8,scroll_events,NaN,NaN,NaN
9,sessions_organic,NaN,NaN,NaN


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verify the Data Contract

The following checks verify the assumptions of the data contract by confirming the dataset grain, validating the analysis period, examining missing values, and ensuring Google Search Console and Google Analytics data are available.

In [9]:
print("Duplicate rows:",
      df.duplicated(
          subset=[
              "report_date",
              "client_hash_id",
              "content_hash_id"
          ]
      ).sum())



Duplicate rows: 0


In [10]:
print("Dataset shape:", df.shape)

print("Start:", df["report_date"].min())

print("End:", df["report_date"].max())

Dataset shape: (9841378, 30)
Start: 2026-03-01 00:00:00
End: 2026-03-31 00:00:00


In [11]:
missing = pd.DataFrame({
    "Missing Count": df[features].isnull().sum(),
    "Missing %": (df[features].isnull().mean()*100).round(2)
})

missing

,Missing Count,Missing %
gsc_impressions,0,0.00
gsc_clicks,0,0.00
gsc_avg_position,6230317,63.31
ga4_pageviews,3018741,30.67
ga4_sessions,3018741,30.67
ga4_users,3018741,30.67
ga4_engaged_sessions,3018741,30.67
ga4_total_engagement_sec,3018741,30.67
scroll_events,3018741,30.67
sessions_organic,3018741,30.67


In [12]:
context_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

df[context_cols].isnull().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0


In [13]:
df[features].describe().T

,count,mean,std,min,25%,50%,75%,max
gsc_impressions,9841378.0,28.518119,155.926569,0.0,0.00000,0.0,6.0,40084.0
gsc_clicks,9841378.0,0.083508,0.781434,0.0,0.00000,0.0,0.0,274.0
gsc_avg_position,3611061.0,15.826651,19.856034,0.0,3.74212,7.5,20.2,498.0
ga4_pageviews,6822637.0,0.217637,2.142851,0.0,0.00000,0.0,0.0,875.0
ga4_sessions,6822637.0,0.190514,1.968750,0.0,0.00000,0.0,0.0,792.0
ga4_users,6822637.0,0.186090,1.961616,0.0,0.00000,0.0,0.0,740.0
ga4_engaged_sessions,6822637.0,0.004331,0.076647,0.0,0.00000,0.0,0.0,21.0
ga4_total_engagement_sec,6822637.0,0.698905,17.206612,0.0,0.00000,0.0,0.0,7083.0
scroll_events,6822637.0,0.032261,0.413166,0.0,0.00000,0.0,0.0,254.0
sessions_organic,6822637.0,0.085804,0.875436,0.0,0.00000,0.0,0.0,405.0


In [14]:
available = df[
    (df["gsc_data_available"]) &
    (df["ga4_data_available"])
]

print("Rows with both data sources:", len(available))

Rows with both data sources: 364347


5. Leakage Awareness (NEW)

The selected features describe the observed state of each content page during the analysis window. No future information or manually constructed target variables are included at this stage. Any proxy target created later will be engineered using only information available within the defined analysis period to minimize data leakage.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This analysis has several important limitations:

The dataset contains observational historical data and therefore cannot establish causal relationships between content signals and search performance.
Some records may contain incomplete Google Search Console or Google Analytics measurements.
The analysis uses only March 2026 and therefore does not capture seasonal or long-term trends.
The dataset does not contain an explicit optimization-priority label, requiring the use of an engineered proxy target.
Client identities, domains, and search queries are anonymized, preventing client-specific analysis while preserving privacy.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.